# Fair Value Gap Analysis (All Data Folder Files)
This notebook replicates your FVG workflow on all CSV files inside the `data` folder.

In [9]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

In [10]:
data_dir = Path('data')
csv_files = sorted(data_dir.glob('*.csv'))

if not csv_files:
    raise FileNotFoundError(f'No CSV files found in {data_dir.resolve()}')

frames = []
for file in csv_files:
    temp = pd.read_csv(file)
    temp.columns = [c.strip() for c in temp.columns]
    temp['Sector'] = file.stem
    frames.append(temp)

raw_df = pd.concat(frames, ignore_index=True)

required_cols = ['Symbol', 'Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Sector']
missing_cols = [c for c in required_cols if c not in raw_df.columns]
if missing_cols:
    raise ValueError(f'Missing required columns across data files: {missing_cols}')

df = raw_df[required_cols].copy()
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.dropna(subset=['Date'])
df = df.sort_values(['Symbol', 'Date']).reset_index(drop=True)

print(f'Loaded {len(csv_files)} CSV files from {data_dir}.')
print(f'Total rows after cleaning: {len(df):,}')
print(f'Unique symbols: {df["Symbol"].nunique()}')
print(f'Unique sectors(files): {df["Sector"].nunique()}')
df.head()

Loaded 13 CSV files from data.
Total rows after cleaning: 15,053
Unique symbols: 13
Unique sectors(files): 13


,Symbol,Date,Open,High,Low,Close,Volume,Sector
0,BANKING,2021-03-31,1768.56,1806.88,1767.60,1806.11,-,banking
1,BANKING,2021-04-01,1818.70,1824.72,1802.87,1811.54,-,banking
2,BANKING,2021-04-04,1831.98,1831.98,1818.84,1826.20,-,banking
3,BANKING,2021-04-05,1845.43,1845.43,1812.71,1814.42,-,banking
4,BANKING,2021-04-06,1820.71,1820.71,1804.10,1813.83,-,banking


In [11]:
# Fair Value Gap (FVG) detection using a 3-candle structure
# Bullish FVG at i: Low[i] > High[i-2]
# Bearish FVG at i: High[i] < Low[i-2]
# Keep only gaps where size >= MIN_GAP_PCT of the reference price

MIN_GAP_PCT = 0.5

fvg_df = df.copy()
records = []

for sym, g in fvg_df.groupby('Symbol', sort=False):
    g = g.sort_values('Date').reset_index(drop=True)

    for i in range(2, len(g)):
        c0 = g.loc[i - 2]
        c2 = g.loc[i]

        # Bullish imbalance gap
        if c2['Low'] > c0['High']:
            gap_size = c2['Low'] - c0['High']
            gap_pct = (gap_size / c0['High']) * 100

            if gap_pct >= MIN_GAP_PCT:
                records.append({
                    'Symbol': sym,
                    'Sector': c2['Sector'],
                    'Date': c2['Date'],
                    'Type': 'Bullish',
                    'Gap_Low': c0['High'],
                    'Gap_High': c2['Low'],
                    'Gap_Size': gap_size,
                    'Gap_Pct': gap_pct,
                    'Bar_Index': i
                })

        # Bearish imbalance gap
        if c2['High'] < c0['Low']:
            gap_size = c0['Low'] - c2['High']
            gap_pct = (gap_size / c0['Low']) * 100

            if gap_pct >= MIN_GAP_PCT:
                records.append({
                    'Symbol': sym,
                    'Sector': c2['Sector'],
                    'Date': c2['Date'],
                    'Type': 'Bearish',
                    'Gap_Low': c2['High'],
                    'Gap_High': c0['Low'],
                    'Gap_Size': gap_size,
                    'Gap_Pct': gap_pct,
                    'Bar_Index': i
                })

fvg = pd.DataFrame(records)

if not fvg.empty:
    fvg = fvg.sort_values(['Symbol', 'Type', 'Bar_Index']).reset_index(drop=True)

    fvg['Is_Continuous'] = (
        (fvg['Symbol'] == fvg['Symbol'].shift(1))
        & (fvg['Type'] == fvg['Type'].shift(1))
        & ((fvg['Bar_Index'] - fvg['Bar_Index'].shift(1)) == 1)
    )

    fvg['Continuous_Group'] = (~fvg['Is_Continuous']).cumsum()

    fvg = (
        fvg.loc[fvg.groupby(['Symbol', 'Type', 'Continuous_Group'])['Gap_Size'].idxmax()]
        .sort_values(['Symbol', 'Date'])
        .reset_index(drop=True)
    )

print(f'Total FVGs found (>= {MIN_GAP_PCT}% and deduped continuous): {len(fvg)}')
fvg.head(20)

Total FVGs found (>= 0.5% and deduped continuous): 2114


,Symbol,Sector,Date,Type,Gap_Low,Gap_High,Gap_Size,Gap_Pct,Bar_Index,Is_Continuous,Continuous_Group
0,BANKING,banking,2021-04-04,Bullish,1806.88,1818.84,11.96,0.661914,2,False,71
1,BANKING,banking,2021-04-15,Bullish,1839.08,1896.43,57.35,3.118407,9,True,72
2,BANKING,banking,2021-04-20,Bearish,1851.86,1870.86,19.00,1.015576,12,True,1
3,BANKING,banking,2021-04-26,Bearish,1783.15,1794.02,10.87,0.605902,16,False,2
4,BANKING,banking,2021-05-24,Bullish,1857.75,1876.93,19.18,1.032432,36,False,73
5,BANKING,banking,2021-06-06,Bullish,1874.48,1960.34,85.86,4.580470,44,False,74
6,BANKING,banking,2021-06-15,Bullish,1951.32,1988.64,37.32,1.912552,51,False,75
7,BANKING,banking,2021-06-28,Bullish,1918.51,1945.79,27.28,1.421937,60,False,76
8,BANKING,banking,2021-07-12,Bullish,1928.95,1940.64,11.69,0.606029,70,False,77
9,BANKING,banking,2021-07-19,Bullish,1968.97,2060.40,91.43,4.643545,75,False,78


In [12]:
# FVG performance analysis setup
horizons = [3, 5, 10, 15, 20]

if fvg.empty:
    raise ValueError('fvg is empty. Run the FVG detection cell first.')

signal_records = []
performance_records = []

for signal_id, sig in fvg.reset_index(drop=True).iterrows():
    sym = sig['Symbol']
    fvg_type = sig['Type']
    gap_low = float(sig['Gap_Low'])
    gap_high = float(sig['Gap_High'])
    sig_idx = int(sig['Bar_Index'])

    sym_df = df[df['Symbol'] == sym].sort_values('Date').reset_index(drop=True)

    if sig_idx + 1 >= len(sym_df):
        continue

    after_signal = sym_df.iloc[sig_idx + 1:].copy()
    touches_gap = (after_signal['Low'] <= gap_high) & (after_signal['High'] >= gap_low)

    if not touches_gap.any():
        continue

    first_touch_offset = np.where(touches_gap.to_numpy())[0][0]
    entry_idx = sig_idx + 1 + first_touch_offset
    entry_row = sym_df.loc[entry_idx]

    entry_date = entry_row['Date']
    entry_price = float(entry_row['Close'])
    sector_value = entry_row['Sector']

    signal_records.append({
        'Signal_ID': signal_id,
        'Symbol': sym,
        'Sector': sector_value,
        'FVG_Type': fvg_type,
        'Signal_Date': sig['Date'],
        'Entry_Date': entry_date,
        'Entry_Index': entry_idx,
        'Entry_Price': entry_price,
        'Gap_Low': gap_low,
        'Gap_High': gap_high,
        'Gap_Size': float(sig['Gap_Size']),
        'Gap_Pct': float(sig.get('Gap_Pct', np.nan))
    })

    for h in horizons:
        future_idx = entry_idx + h
        if future_idx >= len(sym_df):
            continue

        future_close = float(sym_df.loc[future_idx, 'Close'])
        path = sym_df.iloc[entry_idx + 1:future_idx + 1]

        if fvg_type == 'Bullish':
            ret_pct = ((future_close - entry_price) / entry_price) * 100
            mfe_pct = ((path['High'].max() - entry_price) / entry_price) * 100
            mae_pct = ((path['Low'].min() - entry_price) / entry_price) * 100
        else:
            ret_pct = ((entry_price - future_close) / entry_price) * 100
            mfe_pct = ((entry_price - path['Low'].min()) / entry_price) * 100
            mae_pct = ((entry_price - path['High'].max()) / entry_price) * 100

        rr = np.nan if mae_pct >= 0 else (mfe_pct / abs(mae_pct))

        performance_records.append({
            'Signal_ID': signal_id,
            'Symbol': sym,
            'Sector': sector_value,
            'FVG_Type': fvg_type,
            'Horizon': h,
            'Return_Pct': ret_pct,
            'MFE_Pct': mfe_pct,
            'MAE_Pct': mae_pct,
            'RR': rr,
            'Win': int(ret_pct > 0),
            'Win_MFE': int(mfe_pct > 0),
            'Win_MAE': int(mae_pct < 0)
        })

entry_points = pd.DataFrame(signal_records)
perf = pd.DataFrame(performance_records)

print(f'Total detected FVG signals: {len(fvg)}')
print(f'Signals with a valid re-entry entry point: {entry_points["Signal_ID"].nunique() if not entry_points.empty else 0}')
print(f'Signals contributing to at least one horizon metric: {perf["Signal_ID"].nunique() if not perf.empty else 0}')

Total detected FVG signals: 2114
Signals with a valid re-entry entry point: 2049
Signals contributing to at least one horizon metric: 2041


In [13]:
if perf.empty:
    print('No valid FVG entries with enough forward candles for the selected horizons.')
else:
    summary_by_type = (
        perf.groupby(['FVG_Type', 'Horizon'], as_index=False)
        .agg(
            Signals=('Return_Pct', 'count'),
            Avg_Return_Pct=('Return_Pct', lambda x: x.mean() * 100),
            Win_Rate_Pct=('Win', lambda x: x.mean() * 100),
            Avg_MFE_Pct=('MFE_Pct', 'mean'),
            Win_Rate_Pct_MFE=('Win_MFE', lambda x: x.mean() * 100),
            Avg_MAE_Pct=('MAE_Pct', 'mean'),
            Win_Rate_Pct_MAE=('Win_MAE', lambda x: x.mean() * 100)
        )
        .sort_values(['FVG_Type', 'Horizon'])
        .reset_index(drop=True)
    )

    summary_by_sector = (
        perf.groupby(['Sector', 'FVG_Type', 'Horizon'], as_index=False)
        .agg(
            Signals=('Return_Pct', 'count'),
            Avg_Return_Pct=('Return_Pct', lambda x: x.mean() * 100),
            Win_Rate_Pct=('Win', lambda x: x.mean() * 100),
            Avg_MFE_Pct=('MFE_Pct', 'mean'),
            Win_Rate_Pct_MFE=('Win_MFE', lambda x: x.mean() * 100),
            Avg_MAE_Pct=('MAE_Pct', 'mean'),
            Win_Rate_Pct_MAE=('Win_MAE', lambda x: x.mean() * 100)
        )
        .sort_values(['Sector', 'FVG_Type', 'Horizon'])
        .reset_index(drop=True)
    )

    # One dataframe per sector, stored in a dictionary.
    sector_dataframes = {
        sector: sec_df.drop(columns=['Sector']).reset_index(drop=True)
        for sector, sec_df in summary_by_sector.groupby('Sector', sort=True)
    }

    print('Performance split by Bullish vs Bearish FVG')
    display(summary_by_type)

    print('Performance by sector (separate dataframe for each sector)')
    for sector_name, sector_df in sector_dataframes.items():
        print(f'Sector: {sector_name}')
        display(sector_df)

    print('Sector dataframe keys available in `sector_dataframes`:')
    print(list(sector_dataframes.keys()))

Performance split by Bullish vs Bearish FVG


,FVG_Type,Horizon,Signals,Avg_Return_Pct,Win_Rate_Pct,Avg_MFE_Pct,Win_Rate_Pct_MFE,Avg_MAE_Pct,Win_Rate_Pct_MAE
0,Bearish,3,978,38.265593,60.940695,2.568634,93.558282,-2.477785,97.239264
1,Bearish,5,978,19.922820,56.032720,3.334652,94.887526,-3.307198,97.648262
2,Bearish,10,975,-18.148218,53.743590,4.801596,95.897436,-5.127223,98.666667
3,Bearish,15,970,-42.899158,53.298969,5.689751,96.907216,-6.585125,99.175258
4,Bearish,20,966,-81.178923,53.209110,6.504539,97.204969,-7.850234,99.275362
5,Bullish,3,1063,15.199158,44.496707,2.803017,96.613358,-2.195016,93.885230
6,Bullish,5,1059,32.188954,43.909348,3.766835,97.072710,-2.927467,95.278565
7,Bullish,10,1055,64.010672,45.402844,5.665349,97.535545,-4.117716,96.303318
8,Bullish,15,1047,58.960068,45.272206,7.066645,97.707736,-5.146425,96.752627
9,Bullish,20,1043,35.162076,46.021093,8.074186,97.890700,-6.044770,97.219559


Performance by sector (separate dataframe for each sector)
Sector: DevBank


,FVG_Type,Horizon,Signals,Avg_Return_Pct,Win_Rate_Pct,Avg_MFE_Pct,Win_Rate_Pct_MFE,Avg_MAE_Pct,Win_Rate_Pct_MAE
0,Bearish,3,84,69.704155,64.285714,2.920615,92.857143,-2.438578,96.428571
1,Bearish,5,84,19.120239,54.761905,3.773751,95.238095,-3.413746,97.619048
2,Bearish,10,83,-77.923381,50.602410,5.100153,96.385542,-5.425619,100.000000
3,Bearish,15,83,-116.697744,57.831325,5.737065,97.590361,-7.317870,100.000000
4,Bearish,20,82,-120.062100,54.878049,6.622671,97.560976,-8.618092,100.000000
5,Bullish,3,101,27.183088,46.534653,3.244050,95.049505,-2.246122,93.069307
6,Bullish,5,101,94.918218,51.485149,4.561394,97.029703,-3.134332,94.059406
7,Bullish,10,100,173.379193,50.000000,7.279626,97.000000,-4.480877,95.000000
8,Bullish,15,100,188.293711,52.000000,9.260980,98.000000,-5.250969,95.000000
9,Bullish,20,99,158.373248,53.535354,10.327717,97.979798,-6.072113,96.969697


Sector: Finance


...

Sector: banking


,FVG_Type,Horizon,Signals,Avg_Return_Pct,Win_Rate_Pct,Avg_MFE_Pct,Win_Rate_Pct_MFE,Avg_MAE_Pct,Win_Rate_Pct_MAE
0,Bearish,3,68,-9.674207,57.352941,1.666675,86.764706,-2.064578,95.588235
1,Bearish,5,68,-4.936051,55.882353,2.267369,88.235294,-2.721315,95.588235
2,Bearish,10,68,27.370212,58.823529,3.322593,89.705882,-3.700400,98.529412
3,Bearish,15,68,65.404094,66.176471,4.169299,89.705882,-4.536319,98.529412
4,Bearish,20,68,77.108804,60.294118,4.772371,92.647059,-5.207175,98.529412
5,Bullish,3,75,26.558473,44.000000,2.195892,97.333333,-1.442400,90.666667
6,Bullish,5,75,19.402583,42.666667,2.811507,97.333333,-1.981300,93.333333
7,Bullish,10,75,-15.561905,40.000000,3.978049,98.666667,-2.952264,94.666667
8,Bullish,15,74,-49.767303,36.486486,4.829491,98.648649,-3.956135,97.297297
9,Bullish,20,74,-53.730557,37.837838,5.390352,98.648649,-4.591377,97.297297


Sector: nepse


,FVG_Type,Horizon,Signals,Avg_Return_Pct,Win_Rate_Pct,Avg_MFE_Pct,Win_Rate_Pct_MFE,Avg_MAE_Pct,Win_Rate_Pct_MAE
0,Bearish,3,89,52.392953,57.303371,2.058926,95.505618,-1.664172,98.876404
1,Bearish,5,89,4.916272,53.932584,2.653417,97.752809,-2.445941,98.876404
2,Bearish,10,89,18.338791,51.685393,3.864890,97.752809,-3.845151,98.876404
3,Bearish,15,89,-4.988884,47.191011,4.626293,100.000000,-4.845497,98.876404
4,Bearish,20,88,-30.172440,50.000000,5.352148,100.000000,-5.526875,98.863636
5,Bullish,3,92,-8.952196,46.739130,1.942421,98.913043,-1.664607,92.391304
6,Bullish,5,92,-10.232483,41.304348,2.499019,98.913043,-2.318587,96.739130
7,Bullish,10,92,-13.079323,40.217391,3.548459,98.913043,-3.267752,98.913043
8,Bullish,15,91,-58.307083,42.857143,4.434303,98.901099,-4.378606,98.901099
9,Bullish,20,91,-78.253612,43.956044,5.151862,98.901099,-5.186530,98.901099


Sector dataframe keys available in `sector_dataframes`:
['DevBank', 'Finance', 'Hotel', 'Hydropower', 'Investment', 'Life Insurance', 'Manu.&Pro.', 'Microfinance', 'Nonlife', 'Others', 'Trading', 'banking', 'nepse']


In [16]:
# Grouped bar charts for all summary dataframes
# X-axis: Horizon
# Left bar: Bullish, Right bar: Bearish
# Bar-top label: Avg_Return_Pct value
# Side label: Win_Rate_Pct

if 'summary_by_type' not in globals() or summary_by_type.empty:
    raise ValueError('Run Cell 6 first so summary dataframes are available.')


def plot_grouped_bars_with_winrate(plot_df, title):
    required = {'Horizon', 'FVG_Type', 'Avg_Return_Pct', 'Win_Rate_Pct'}
    missing = required - set(plot_df.columns)
    if missing:
        print(f"Skipped {title}: missing columns {sorted(missing)}")
        return

    local_df = plot_df.copy()
    local_df['FVG_Type'] = pd.Categorical(
        local_df['FVG_Type'], categories=['Bullish', 'Bearish'], ordered=True
    )

    horizons_sorted = sorted(local_df['Horizon'].dropna().unique())
    full_idx = pd.MultiIndex.from_product(
        [horizons_sorted, ['Bullish', 'Bearish']],
        names=['Horizon', 'FVG_Type']
    )

    local_df = (
        local_df.set_index(['Horizon', 'FVG_Type'])
        .reindex(full_idx)
        .reset_index()
    )

    local_df['Avg_Return_Pct'] = local_df['Avg_Return_Pct'].fillna(0.0)
    local_df['Win_Rate_Pct'] = local_df['Win_Rate_Pct'].fillna(0.0)
    local_df['Horizon_Str'] = local_df['Horizon'].astype(int).astype(str)

    bull = local_df[local_df['FVG_Type'] == 'Bullish'].copy()
    bear = local_df[local_df['FVG_Type'] == 'Bearish'].copy()

    fig = go.Figure()

    fig.add_trace(
        go.Bar(
            x=bull['Horizon_Str'],
            y=bull['Avg_Return_Pct'],
            name='Bullish',
            marker_color='rgba(34, 139, 34, 0.9)',
            text=[f"{v:.2f}%" for v in bull['Avg_Return_Pct']],
            textposition='outside',
            cliponaxis=False,
            offsetgroup='Bullish'
        )
    )

    fig.add_trace(
        go.Bar(
            x=bear['Horizon_Str'],
            y=bear['Avg_Return_Pct'],
            name='Bearish',
            marker_color='rgba(220, 20, 60, 0.9)',
            text=[f"{v:.2f}%" for v in bear['Avg_Return_Pct']],
            textposition='outside',
            cliponaxis=False,
            offsetgroup='Bearish'
        )
    )

    # Win labels as separate text traces for stable left/right placement.
    fig.add_trace(
        go.Scatter(
            x=bull['Horizon_Str'],
            y=bull['Avg_Return_Pct'] / 2,
            mode='text',
            text=[f"Win {v:.1f}%" for v in bull['Win_Rate_Pct']],
            textposition='middle left',
            textfont=dict(size=11, color='black'),
            showlegend=False,
            hoverinfo='skip'
        )
    )

    fig.add_trace(
        go.Scatter(
            x=bear['Horizon_Str'],
            y=bear['Avg_Return_Pct'] / 2,
            mode='text',
            text=[f"Win {v:.1f}%" for v in bear['Win_Rate_Pct']],
            textposition='middle right',
            textfont=dict(size=11, color='black'),
            showlegend=False,
            hoverinfo='skip'
        )
    )

    y_max = float(local_df['Avg_Return_Pct'].max()) if not local_df.empty else 1.0
    y_min = float(local_df['Avg_Return_Pct'].min()) if not local_df.empty else -1.0
    span = max(abs(y_max), abs(y_min), 1.0)

    fig.update_layout(
        title=title,
        barmode='group',
        bargap=0.25,
        bargroupgap=0.08,
        xaxis_title='Horizon',
        yaxis_title='Avg Return %',
        template='plotly_white',
        height=560,
        legend_title='FVG Type'
    )

    fig.update_yaxes(range=[-span * 1.2, span * 1.2])
    fig.show()


# Plot for overall Bullish vs Bearish summary
plot_grouped_bars_with_winrate(summary_by_type, 'All Sectors: Bullish vs Bearish by Horizon')

# Plot for each sector dataframe
if 'sector_dataframes' in globals() and sector_dataframes:
    for sector_name in sorted(sector_dataframes.keys()):
        sector_df = sector_dataframes[sector_name].copy()
        plot_grouped_bars_with_winrate(sector_df, f"Sector: {sector_name} | Bullish vs Bearish by Horizon")
else:
    print('sector_dataframes is missing or empty. Run Cell 6 again.')

In [17]:
# Grouped bar charts for MAE across all summary dataframes
# X-axis: Horizon
# Left bar: Bullish, Right bar: Bearish
# Bar-top label: Avg_MAE_Pct value
# Side label: Win_Rate_Pct_MAE

if 'summary_by_type' not in globals() or summary_by_type.empty:
    raise ValueError('Run Cell 6 first so summary dataframes are available.')


def plot_grouped_mae_with_winrate(plot_df, title):
    required = {'Horizon', 'FVG_Type', 'Avg_MAE_Pct', 'Win_Rate_Pct_MAE'}
    missing = required - set(plot_df.columns)
    if missing:
        print(f"Skipped {title}: missing columns {sorted(missing)}")
        return

    local_df = plot_df.copy()
    local_df['FVG_Type'] = pd.Categorical(
        local_df['FVG_Type'], categories=['Bullish', 'Bearish'], ordered=True
    )

    horizons_sorted = sorted(local_df['Horizon'].dropna().unique())
    full_idx = pd.MultiIndex.from_product(
        [horizons_sorted, ['Bullish', 'Bearish']],
        names=['Horizon', 'FVG_Type']
    )

    local_df = (
        local_df.set_index(['Horizon', 'FVG_Type'])
        .reindex(full_idx)
        .reset_index()
    )

    local_df['Avg_MAE_Pct'] = local_df['Avg_MAE_Pct'].fillna(0.0)
    local_df['Win_Rate_Pct_MAE'] = local_df['Win_Rate_Pct_MAE'].fillna(0.0)
    local_df['Horizon_Str'] = local_df['Horizon'].astype(int).astype(str)

    bull = local_df[local_df['FVG_Type'] == 'Bullish'].copy()
    bear = local_df[local_df['FVG_Type'] == 'Bearish'].copy()

    fig = go.Figure()

    fig.add_trace(
        go.Bar(
            x=bull['Horizon_Str'],
            y=bull['Avg_MAE_Pct'],
            name='Bullish',
            marker_color='rgba(34, 139, 34, 0.9)',
            text=[f"{v:.2f}%" for v in bull['Avg_MAE_Pct']],
            textposition='outside',
            cliponaxis=False,
            offsetgroup='Bullish'
        )
    )

    fig.add_trace(
        go.Bar(
            x=bear['Horizon_Str'],
            y=bear['Avg_MAE_Pct'],
            name='Bearish',
            marker_color='rgba(220, 20, 60, 0.9)',
            text=[f"{v:.2f}%" for v in bear['Avg_MAE_Pct']],
            textposition='outside',
            cliponaxis=False,
            offsetgroup='Bearish'
        )
    )

    fig.add_trace(
        go.Scatter(
            x=bull['Horizon_Str'],
            y=bull['Avg_MAE_Pct'] / 2,
            mode='text',
            text=[f"Win {v:.1f}%" for v in bull['Win_Rate_Pct_MAE']],
            textposition='middle left',
            textfont=dict(size=11, color='black'),
            showlegend=False,
            hoverinfo='skip'
        )
    )

    fig.add_trace(
        go.Scatter(
            x=bear['Horizon_Str'],
            y=bear['Avg_MAE_Pct'] / 2,
            mode='text',
            text=[f"Win {v:.1f}%" for v in bear['Win_Rate_Pct_MAE']],
            textposition='middle right',
            textfont=dict(size=11, color='black'),
            showlegend=False,
            hoverinfo='skip'
        )
    )

    y_max = float(local_df['Avg_MAE_Pct'].max()) if not local_df.empty else 1.0
    y_min = float(local_df['Avg_MAE_Pct'].min()) if not local_df.empty else -1.0
    span = max(abs(y_max), abs(y_min), 1.0)

    fig.update_layout(
        title=title,
        barmode='group',
        bargap=0.25,
        bargroupgap=0.08,
        xaxis_title='Horizon',
        yaxis_title='Avg MAE %',
        template='plotly_white',
        height=560,
        legend_title='FVG Type'
    )

    fig.update_yaxes(range=[-span * 1.2, span * 1.2])
    fig.show()


plot_grouped_mae_with_winrate(summary_by_type, 'All Sectors: MAE by Horizon')

if 'sector_dataframes' in globals() and sector_dataframes:
    for sector_name in sorted(sector_dataframes.keys()):
        sector_df = sector_dataframes[sector_name].copy()
        plot_grouped_mae_with_winrate(sector_df, f"Sector: {sector_name} | MAE by Horizon")
else:
    print('sector_dataframes is missing or empty. Run Cell 6 again.')

In [18]:
# Grouped bar charts for MFE across all summary dataframes
# X-axis: Horizon
# Left bar: Bullish, Right bar: Bearish
# Bar-top label: Avg_MFE_Pct value
# Side label: Win_Rate_Pct_MFE

if 'summary_by_type' not in globals() or summary_by_type.empty:
    raise ValueError('Run Cell 6 first so summary dataframes are available.')


def plot_grouped_mfe_with_winrate(plot_df, title):
    required = {'Horizon', 'FVG_Type', 'Avg_MFE_Pct', 'Win_Rate_Pct_MFE'}
    missing = required - set(plot_df.columns)
    if missing:
        print(f"Skipped {title}: missing columns {sorted(missing)}")
        return

    local_df = plot_df.copy()
    local_df['FVG_Type'] = pd.Categorical(
        local_df['FVG_Type'], categories=['Bullish', 'Bearish'], ordered=True
    )

    horizons_sorted = sorted(local_df['Horizon'].dropna().unique())
    full_idx = pd.MultiIndex.from_product(
        [horizons_sorted, ['Bullish', 'Bearish']],
        names=['Horizon', 'FVG_Type']
    )

    local_df = (
        local_df.set_index(['Horizon', 'FVG_Type'])
        .reindex(full_idx)
        .reset_index()
    )

    local_df['Avg_MFE_Pct'] = local_df['Avg_MFE_Pct'].fillna(0.0)
    local_df['Win_Rate_Pct_MFE'] = local_df['Win_Rate_Pct_MFE'].fillna(0.0)
    local_df['Horizon_Str'] = local_df['Horizon'].astype(int).astype(str)

    bull = local_df[local_df['FVG_Type'] == 'Bullish'].copy()
    bear = local_df[local_df['FVG_Type'] == 'Bearish'].copy()

    fig = go.Figure()

    fig.add_trace(
        go.Bar(
            x=bull['Horizon_Str'],
            y=bull['Avg_MFE_Pct'],
            name='Bullish',
            marker_color='rgba(34, 139, 34, 0.9)',
            text=[f"{v:.2f}%" for v in bull['Avg_MFE_Pct']],
            textposition='outside',
            cliponaxis=False,
            offsetgroup='Bullish'
        )
    )

    fig.add_trace(
        go.Bar(
            x=bear['Horizon_Str'],
            y=bear['Avg_MFE_Pct'],
            name='Bearish',
            marker_color='rgba(220, 20, 60, 0.9)',
            text=[f"{v:.2f}%" for v in bear['Avg_MFE_Pct']],
            textposition='outside',
            cliponaxis=False,
            offsetgroup='Bearish'
        )
    )

    fig.add_trace(
        go.Scatter(
            x=bull['Horizon_Str'],
            y=bull['Avg_MFE_Pct'] / 2,
            mode='text',
            text=[f"Win {v:.1f}%" for v in bull['Win_Rate_Pct_MFE']],
            textposition='middle left',
            textfont=dict(size=11, color='black'),
            showlegend=False,
            hoverinfo='skip'
        )
    )

    fig.add_trace(
        go.Scatter(
            x=bear['Horizon_Str'],
            y=bear['Avg_MFE_Pct'] / 2,
            mode='text',
            text=[f"Win {v:.1f}%" for v in bear['Win_Rate_Pct_MFE']],
            textposition='middle right',
            textfont=dict(size=11, color='black'),
            showlegend=False,
            hoverinfo='skip'
        )
    )

    y_max = float(local_df['Avg_MFE_Pct'].max()) if not local_df.empty else 1.0
    y_min = float(local_df['Avg_MFE_Pct'].min()) if not local_df.empty else -1.0
    span = max(abs(y_max), abs(y_min), 1.0)

    fig.update_layout(
        title=title,
        barmode='group',
        bargap=0.25,
        bargroupgap=0.08,
        xaxis_title='Horizon',
        yaxis_title='Avg MFE %',
        template='plotly_white',
        height=560,
        legend_title='FVG Type'
    )

    fig.update_yaxes(range=[-span * 1.2, span * 1.2])
    fig.show()


plot_grouped_mfe_with_winrate(summary_by_type, 'All Sectors: MFE by Horizon')

if 'sector_dataframes' in globals() and sector_dataframes:
    for sector_name in sorted(sector_dataframes.keys()):
        sector_df = sector_dataframes[sector_name].copy()
        plot_grouped_mfe_with_winrate(sector_df, f"Sector: {sector_name} | MFE by Horizon")
else:
    print('sector_dataframes is missing or empty. Run Cell 6 again.')

In [14]:
# Optional visualization for one selected symbol
if df.empty:
    print('No data loaded.')
else:
    symbol = df['Symbol'].iloc[0]
    plot_df = df[df['Symbol'] == symbol].sort_values('Date').copy()

    fig = go.Figure(
        data=[
            go.Candlestick(
                x=plot_df['Date'],
                open=plot_df['Open'],
                high=plot_df['High'],
                low=plot_df['Low'],
                close=plot_df['Close'],
                name=symbol,
                increasing_line_color='green',
                decreasing_line_color='red'
            )
        ]
    )

    symbol_fvg = fvg[fvg['Symbol'] == symbol].copy() if 'fvg' in globals() else pd.DataFrame()

    if not symbol_fvg.empty:
        bullish = symbol_fvg[symbol_fvg['Type'] == 'Bullish']
        bearish = symbol_fvg[symbol_fvg['Type'] == 'Bearish']

        if not bullish.empty:
            fig.add_trace(
                go.Scatter(
                    x=bullish['Date'],
                    y=bullish['Gap_High'],
                    mode='markers',
                    name='Bullish FVG',
                    marker=dict(symbol='triangle-up', size=8, color='rgba(34, 139, 34, 0.9)'),
                    customdata=np.stack([bullish['Gap_Low'], bullish['Gap_High'], bullish['Gap_Size']], axis=-1),
                    hovertemplate='Date: %{x}<br>Type: Bullish FVG<br>Gap Low: %{customdata[0]:.2f}<br>Gap High: %{customdata[1]:.2f}<br>Gap Size: %{customdata[2]:.2f}<extra></extra>'
                )
            )

        if not bearish.empty:
            fig.add_trace(
                go.Scatter(
                    x=bearish['Date'],
                    y=bearish['Gap_Low'],
                    mode='markers',
                    name='Bearish FVG',
                    marker=dict(symbol='triangle-down', size=8, color='rgba(220, 20, 60, 0.9)'),
                    customdata=np.stack([bearish['Gap_Low'], bearish['Gap_High'], bearish['Gap_Size']], axis=-1),
                    hovertemplate='Date: %{x}<br>Type: Bearish FVG<br>Gap Low: %{customdata[0]:.2f}<br>Gap High: %{customdata[1]:.2f}<br>Gap Size: %{customdata[2]:.2f}<extra></extra>'
                )
            )

    fig.update_layout(
        title=f'{symbol} Candlestick with FVG Pointers',
        xaxis_title='Date',
        yaxis_title='Price',
        xaxis_rangeslider_visible=False,
        template='plotly_white',
        height=700
    )

    fig.show()